**Link repo của tôi: https://github.com/ducphat1509-beep/artificial-intelligence**

# Báo cáo Bài tập tuần - Buổi 7
**Nguyễn Đức Phát - 24110296**

---

## 1. Vấn đề bài toán (8-Puzzle Problem)
8-Puzzle là bài toán sắp xếp 8 ô số trên bảng 3x3 chứa các số từ 1 đến 8 và một ô trống (biểu diễn bằng số 0). Mục tiêu là di chuyển ô trống để đưa trạng thái bắt đầu (Start State) về trạng thái đích (Goal State) với số bước đi tối ưu.

## 2. Ý tưởng của thuật toán UCS (Uniform Cost Search) với Manhattan Distance Heuristic
Trong bài học này, ta thực hiện tìm kiếm chi phí đồng nhất (UCS) nhưng có sự cải tiến về hàm chi phí:
- **Hàm chi phí (Cost Function)**: Được tính bằng **Khoảng cách Manhattan (Manhattan Distance)** của trạng thái hiện tại so với trạng thái đích (Goal State). Khoảng cách Manhattan của một ô số là tổng khoảng cách theo hàng và cột từ vị trí hiện tại của nó đến vị trí đúng của nó tại Goal State (không tính ô trống số 0).
- **Cơ chế hoạt động**:
  - Mỗi khi sinh ra các trạng thái kế tiếp (successor states), ta tính toán chi phí Manhattan distance cho từng successor ngay lập tức.
  - Đẩy các successor này cùng chi phí của chúng vào Frontier.
  - Frontier hoạt động như một Priority Queue (Hàng đợi ưu tiên), luôn đảm bảo sắp xếp các trạng thái sao cho trạng thái có chi phí (cost) thấp nhất nằm ở đầu hàng đợi để được lấy ra duyệt trước (`pop(0)`).


In [9]:
# =========================================================================
# CHƯƠNG TRÌNH GIẢI BÀI TOÁN 8-PUZZLE BẰNG THUẬT TOÁN UNIFORM COST SEARCH (UCS)
# - Sử dụng khoảng cách Manhattan (Manhattan Distance) làm hàm chi phí.
# - Successor sinh ra được tính toán chi phí trước rồi mới đẩy vào frontier.
# - Sau khi đẩy vào frontier, đảm bảo frontier được sắp xếp cost thấp nhất lên đầu.
# - Trực quan hóa thời gian thực (Real-time) từng bước thông qua Tkinter.
# =========================================================================

import tkinter as tk

START_STATE = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
GOAL_STATE  = ((1, 2, 3), (4, 5, 6), (7, 8, 0))
MOVES = [
    (-1, 0, 'Up'),
    (1, 0, 'Down'),
    (0, -1, 'Left'),
    (0, 1, 'Right')
]

# ── helpers ──────────────────────────────────────────────
def get_neighbors(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                x, y = i, j
    result = []
    for dx, dy, act in MOVES:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            s = [list(r) for r in state]
            s[x][y], s[nx][ny] = s[nx][ny], s[x][y]
            result.append((tuple(tuple(r) for r in s), act))
    return result

def reconstruct(node):
    path = []
    while node:
        path.append(node['state'])
        node = node['parent']
    return path[::-1]

def manhattan_distance(state, goal):
    dist = 0
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)
            
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                tr, tc = goal_pos[val]
                dist += abs(r - tr) + abs(c - tc)
    return dist

# ── UCS generator ────────────────────────────────────────
def ucs_gen(start, goal):
    start_cost = manhattan_distance(start, goal)
    queue = [{'state': start, 'parent': None, 'cost': start_cost}]
    visited = set()
    expanded = 0
    generated = 0
    
    while queue:
        # Đảm bảo frontier luôn được sắp xếp cost thấp nhất lên đầu trước khi lấy ra duyệt
        queue.sort(key=lambda x: x['cost'])
        node = queue.pop(0)
        state = node['state']
        
        if state in visited:
            continue
        visited.add(state)
        expanded += 1
        
        frontier = [n['state'] for n in queue]
        yield state, frontier, list(visited), expanded, generated, None
        
        if state == goal:
            yield state, [], list(visited), expanded, generated, reconstruct(node)
            return
            
        for nb, _ in get_neighbors(state):
            if nb not in visited:
                nb_cost = manhattan_distance(nb, goal)
                queue.append({'state': nb, 'parent': node, 'cost': nb_cost})
                generated += 1
                
                # Sắp xếp frontier sau khi đẩy successor mới vào để đảm bảo cost thấp nhất lên đầu
                queue.sort(key=lambda x: x['cost'])

# ── Scrollable Frame ─────────────────────────────────────
class ScrollableFrame(tk.Frame):
    def __init__(self, parent, height=180):
        super().__init__(parent)
        self.h_scroll = tk.Scrollbar(self, orient='horizontal')
        self.h_scroll.pack(side='bottom', fill='x')
        self.canvas = tk.Canvas(self, height=height)
        self.canvas.pack(side='left', fill='both', expand=True)
        self.h_scroll.config(command=self.canvas.xview)
        self.canvas.configure(xscrollcommand=self.h_scroll.set)
        self.inner = tk.Frame(self.canvas)
        self.inner.bind('<Configure>', lambda e: self.canvas.configure(scrollregion=self.canvas.bbox('all')))
        self.canvas.create_window((0, 0), window=self.inner, anchor='nw')
        self.canvas.bind('<Shift-MouseWheel>', lambda e: self.canvas.xview_scroll(int(-1 * (e.delta / 120)), 'units'))

# ── GUI ──────────────────────────────────────────────────
class PuzzleGUI:
    def __init__(self, root, gen, title='A* Search'):
        self.root = root
        self.gen  = gen
        self.running = False
        self.step_count = 0
        root.title(f'8 Puzzle {title} – Real-time Visualization')
        sw, sh = root.winfo_screenwidth(), root.winfo_screenheight()
        w, h = min(1400, int(sw * .85)), min(900, int(sh * .85))
        root.geometry(f'{w}x{h}+{(sw-w)//2}+{(sh-h)//2}')
        root.minsize(900, 600)

        self.info = tk.Label(root, text='Press Auto Run or Next Step', font=('Arial', 14, 'bold'))
        self.info.pack(pady=6)

        # 1. Khung điều khiển bf được pack sát dưới cùng TRƯỚC tiên để không bao giờ bị đè hoặc ẩn
        bf = tk.Frame(root)
        bf.pack(side='bottom', fill='x', pady=8)
        
        # Khung con giúp căn giữa các nút bấm ở thanh dưới
        center_f = tk.Frame(bf)
        center_f.pack(anchor='center')
        
        tk.Button(center_f, text='Next Step', width=14, command=self.next_step).pack(side='left', padx=5)
        self.auto_btn = tk.Button(center_f, text='Auto Run', width=14, command=self.toggle_auto)
        self.auto_btn.pack(side='left', padx=5)

        self.speed_var = tk.IntVar(value=300)
        tk.Label(center_f, text='Speed (ms):').pack(side='left', padx=(15, 2))
        tk.Scale(center_f, from_=10, to=1000, orient='horizontal', variable=self.speed_var, showvalue=True, length=160).pack(side='left')

        # 2. Khung trạng thái hiện tại (Popped Node) pack ở trên cùng
        self.cur_frame = tk.LabelFrame(root, text='Current State (Popped Node)', padx=8, pady=8)
        self.cur_frame.pack(side='top', fill='x', padx=10, pady=4)

        # 3. Khung Frontier và Explored sử dụng expand=True để co giãn tự động trong phần không gian ở giữa
        fl = tk.LabelFrame(root, text='Frontier (Priority Queue - Ordered by Lowest Cost First)', padx=8, pady=8)
        fl.pack(side='top', fill='both', expand=True, padx=10, pady=4)
        self.frontier_sf = ScrollableFrame(fl, 180)
        self.frontier_sf.pack(fill='both', expand=True)

        el = tk.LabelFrame(root, text='Explored Set', padx=8, pady=8)
        el.pack(side='top', fill='both', expand=True, padx=10, pady=4)
        self.explored_sf = ScrollableFrame(el, 180)
        self.explored_sf.pack(fill='both', expand=True)

    def draw_state(self, parent, state, label_text=None):
        container = tk.Frame(parent)
        
        grid_f = tk.Frame(container)
        grid_f.pack()
        for i in range(3):
            for j in range(3):
                v = state[i][j]
                tk.Label(grid_f, text=str(v) if v else '', width=4, height=2,
                         font=('Arial', 13, 'bold'), relief='solid',
                         bg='#4CAF50' if v == 0 else 'white').grid(row=i, column=j)
                         
        if label_text:
            lbl = tk.Label(container, text=label_text, font=('Arial', 10, 'bold'), fg='blue')
            lbl.pack(pady=4)
            
        return container

    def clear(self, frame):
        for w in frame.winfo_children(): w.destroy()

    def render(self, current, frontier, explored, expanded, generated, path):
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)
        self.clear(self.explored_sf.inner)

        # Hiển thị popped state kèm theo chi phí f(n)
        cost = manhattan_distance(current, GOAL_STATE)
        self.draw_state(self.cur_frame, current, f'Cost f(n): {cost}').pack()
        
        for i, s in enumerate(frontier):
            f_cost = manhattan_distance(s, GOAL_STATE)
            self.draw_state(self.frontier_sf.inner, s, f'Cost f(n): {f_cost}').grid(row=0, column=i, padx=8, pady=8)
            
        for i, s in enumerate(explored):
            self.draw_state(self.explored_sf.inner, s, 'Explored').grid(row=0, column=i, padx=8, pady=8)

        status = f'Step {self.step_count} | Expanded: {expanded} | Generated: {generated}'
        if path:
            status += f' | GOAL FOUND! Path length: {len(path)-1}'
        self.info.config(text=status)

    def next_step(self):
        try:
            current, frontier, explored, expanded, generated, path = next(self.gen)
            self.step_count += 1
            self.render(current, frontier, explored, expanded, generated, path)
            if path:
                self.running = False
                self.auto_btn.config(text='Auto Run')
        except StopIteration:
            self.running = False
            self.auto_btn.config(text='Done')

    def toggle_auto(self):
        self.running = not self.running
        self.auto_btn.config(text='Pause' if self.running else 'Auto Run')
        if self.running:
            self.auto_tick()

    def auto_tick(self):
        if not self.running: return
        self.next_step()
        if self.running:
            self.root.after(self.speed_var.get(), self.auto_tick)

root = tk.Tk()
app = PuzzleGUI(root, ucs_gen(START_STATE, GOAL_STATE), 'A* Search')
root.mainloop()